# Normalising the prediction errors

1. Read the pixel removal result file
2. Normalise the absolute error in prediction by the area; or use absolute error in prediction
3. Find the pixel number for which, for a given subject the error is maximum.
4. Look at the results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

import sys
sys.path.append("../scripts")

from error_norm import plot_max_error_pixel, read_metadata, MID, YOUNG

%load_ext autoreload
%autoreload 2

output_dir = "../output/removal_data_repl_interpol_6k"
plots_dir = "../output/data_removal_plots"

Path(plots_dir).mkdir(parents=True, exist_ok=True)

* 4096 pixels = 10 seconds
* 409.6 pixels = 1 second

* 1 hz = 1 per second
* 7 hz = 1 / 7 per second = 409.6 / 7 pixels


In [ ]:
409.6 / 15

In [ ]:
avg_arr = np.load("../data/one_beat_array.npy")
chan0_avg = avg_arr[:, :, 0].mean(axis=0)
del avg_arr

In [ ]:
df_meta = read_metadata()

young_age = (df_meta['age'] <= YOUNG)
mid_age = (df_meta['age'] > YOUNG) & (df_meta['age'] <= MID)
old_age = (df_meta['age'] > MID)


In [ ]:
plot_max_error_pixel(
        pixels_removed=14,
        df_meta=df_meta,
        chan0_avg=chan0_avg,
        save_plot=False,
        is_age_group=False,
)

In [ ]:
plot_max_error_pixel(
    pixels_removed=14,
    df_meta=df_meta,
    chan0_avg=chan0_avg,
    save_plot=False,
    is_age_group=True,
)

In [ ]:
df_meta.head()

In [ ]:
df_meta['age'].hist()

In [ ]:
# young-young
((df_meta['age_label'] == 'young') & (df_meta['pred_label'] == 'young')).sum()

In [ ]:
df_meta[young_age]['pred_label'].value_counts()

In [ ]:
df_meta[old_age]['pred_label'].value_counts()

In [ ]:
pixels_removed = 14
filename = f"{output_dir}/all_subjects_and_pixels_{pixels_removed}pixels_6k_sub.csv"
p = Path(filename)
if p.exists():
    df = pd.read_csv(filename)
else:
    raise FileNotFoundError(f"{filename} not found. Run the data removal script first.")
df.loc[:, 'norm_error'] = abs(
    df['raw_prediction'] - df['replace_prediction']) / (df['replace_area'] + 0.0001)

df.loc[:, 'abs_error'] = abs(df['raw_prediction'] - df['replace_prediction']) 



## Normalized Error

Comparison between 2 groups - young age with young prediction and old age with old prediction

In [ ]:
# Per subject what pixel gives maximum norm_error
max_err_pixel = df.loc[
    df.groupby('subject')['norm_error'].idxmax(),
    ['subject', 'start_pixel']
].reset_index(drop=True)
max_err_pixel.loc[:, 'age_label'] = df_meta['age_label']
max_err_pixel.loc[:, 'pred_label'] = df_meta['pred_label']

young_young_pixel = max_err_pixel[
    (max_err_pixel['age_label'] == 'young') &
    (max_err_pixel['pred_label'] == 'young')
]['start_pixel'].values

old_old_pixel = max_err_pixel[
    (max_err_pixel['age_label'] == 'old') &
    (max_err_pixel['pred_label'] == 'old')
]['start_pixel'].values

statistic, p_value = stats.ttest_ind(young_young_pixel, old_old_pixel)
float(p_value)

In [ ]:
MIN_PIX = 1900
MAX_PIX = 2250

fig, ax1 = plt.subplots()
n_bins = MAX_PIX - MIN_PIX

color = 'tab:red'
ax1.set_xlabel(f'Start Pixel for Removing {pixels_removed} pixels')
ax1.set_ylabel('Counts', color=color)
ax1.hist(
    max_err_pixel[
        (max_err_pixel['pred_label'] == 'young') & young_age
    ]['start_pixel'],
    bins=n_bins, color='green', alpha=0.99, label='young-young',
)
ax1.hist(
    max_err_pixel[
        (max_err_pixel['pred_label'] == 'old') & old_age
     ]['start_pixel'],
    bins=n_bins, color='red', alpha=0.2, label='old-old',
)

ax1.legend(loc='upper left')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis
color = 'tab:blue'
ax2.set_ylabel('Average Chan-0 6k', color=color)  # we already handled the x-label with ax1
pixels = [pix for pix in range(MIN_PIX, MAX_PIX)]
ax2.plot(pixels, chan0_avg[MIN_PIX: MAX_PIX], color=color, label='Avg Channel 0')
ax2.tick_params(axis='y', labelcolor=color)
ax2.legend(loc='upper right')
plt.title("Normalized Error")
fig.tight_layout()

## Absolute Error

Comparison between 2 groups - young age with young prediction and old age with old prediction

In [ ]:
# Per subject what pixel gives maximum norm_error
max_err_pixel = df.loc[
    df.groupby('subject')['abs_error'].idxmax(),
    ['subject', 'start_pixel']
].reset_index(drop=True)
max_err_pixel.loc[:, 'age_label'] = df_meta['age_label']
max_err_pixel.loc[:, 'pred_label'] = df_meta['pred_label']

young_young_pixel = max_err_pixel[
    (max_err_pixel['age_label'] == 'young') &
    (max_err_pixel['pred_label'] == 'young')
]['start_pixel'].values

old_old_pixel = max_err_pixel[
    (max_err_pixel['age_label'] == 'old') &
    (max_err_pixel['pred_label'] == 'old')
]['start_pixel'].values

statistic, p_value = stats.ttest_ind(young_young_pixel, old_old_pixel)
float(p_value)

In [ ]:
MIN_PIX = 1900
MAX_PIX = 2250

fig, ax1 = plt.subplots()
n_bins = MAX_PIX - MIN_PIX

color = 'tab:red'
ax1.set_xlabel(f'Start Pixel for Removing {pixels_removed} pixels')
ax1.set_ylabel('Counts', color=color)
ax1.hist(
    max_err_pixel[
        (max_err_pixel['pred_label'] == 'young') & young_age
    ]['start_pixel'],
    bins=n_bins, color='green', alpha=0.9, label='young-young',
)
ax1.hist(
    max_err_pixel[
        (max_err_pixel['pred_label'] == 'old') & old_age
     ]['start_pixel'],
    bins=n_bins, color='red', alpha=0.2, label='old-old',
)

ax1.legend(loc='upper left')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis
color = 'tab:blue'
ax2.set_ylabel('Average Chan-0 6k', color=color)  # we already handled the x-label with ax1
pixels = [pix for pix in range(MIN_PIX, MAX_PIX)]
ax2.plot(pixels, chan0_avg[MIN_PIX: MAX_PIX], color=color, label='Avg Channel 0')
ax2.tick_params(axis='y', labelcolor=color)
ax2.legend(loc='upper right')
plt.title("Absolute Error")
fig.tight_layout()